# Medical AI Text Detection: GPU Embedding EDA

This standalone Colab notebook uses a sentence-transformer on GPU to inspect the
semantic geometry of HC3 Medicine and MedlinePlus text. It visualizes embeddings,
measures group separation, finds nearest neighbors, and tests whether an embedding
classifier trained on HC3 transfers to MedlinePlus.

In Colab select **Runtime → Change runtime type → T4 GPU** before running all cells.

In [ ]:
%pip -q install sentence-transformers umap-learn seaborn scikit-learn

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import umap
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: no GPU detected; embeddings will run on CPU.")

OUTPUT_DIR = Path("/content/embedding_eda_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

## Load the project datasets

Choose one data mode below:

- `upload`: upload the three JSONL files from your computer.
- `drive`: read them from Google Drive.
- `local`: use a checked-out repository under `/content/AI_Detection`.

Required files:

- `hc3_medicine.jsonl`
- `medlineplus_human.jsonl`
- `medlineplus_ai.jsonl`

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA_MODE = "upload"  # "upload", "drive", or "local"

FILES = {
    "hc3": "hc3_medicine.jsonl",
    "medlineplus_human": "medlineplus_human.jsonl",
    "medlineplus_ai": "medlineplus_ai.jsonl",
}

if DATA_MODE == "upload":
    from google.colab import files

    upload_dir = Path("/content/ai_detection_data")
    upload_dir.mkdir(parents=True, exist_ok=True)
    print("Select all three JSONL files.")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        (upload_dir / Path(filename).name).write_bytes(content)
    data_dir = upload_dir
elif DATA_MODE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    # Change this if your project is stored elsewhere in Drive.
    data_dir = Path("/content/drive/MyDrive/AI_Detection/data/processed")
elif DATA_MODE == "local":
    data_dir = Path("/content/AI_Detection/data/processed")
else:
    raise ValueError(f"Unsupported DATA_MODE: {DATA_MODE}")

paths = {name: data_dir / filename for name, filename in FILES.items()}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

print("Data files:")
for name, path in paths.items():
    print(f"  {name:22s} {path}")

In [ ]:
def load_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def records_to_frame(records, dataset_group):
    frame = pd.DataFrame(records)
    frame["dataset_group"] = dataset_group
    return frame


hc3 = records_to_frame(load_jsonl(paths["hc3"]), "HC3")
mp_human = records_to_frame(
    load_jsonl(paths["medlineplus_human"]), "MedlinePlus Human"
)
mp_ai = records_to_frame(
    load_jsonl(paths["medlineplus_ai"]), "MedlinePlus AI"
)

df = pd.concat([hc3, mp_human, mp_ai], ignore_index=True, sort=False)
df["analysis_group"] = df["dataset_group"]
hc3_mask = df["dataset_group"].eq("HC3")
hc3_label_names = df.loc[hc3_mask, "label"].map(
    {"human": "Human", "ai": "AI"}
)
if hc3_label_names.isna().any():
    unknown_index = hc3_label_names.index[hc3_label_names.isna()]
    unknown_labels = sorted(df.loc[unknown_index, "label"].unique())
    raise ValueError(f"Unknown HC3 labels: {unknown_labels}")
df.loc[hc3_mask, "analysis_group"] = "HC3 " + hc3_label_names

print(f"Combined rows: {len(df):,}")
display(df.groupby(["analysis_group", "label"]).size().rename("rows").to_frame())
display(df.head(3))

## 1. Build a balanced exploratory sample

In [ ]:
MAX_PER_GROUP = 500

sampled_parts = []
for group, subset in df.groupby("analysis_group"):
    n = min(len(subset), MAX_PER_GROUP)
    sampled_parts.append(subset.sample(n=n, random_state=RANDOM_STATE))

sample = pd.concat(sampled_parts, ignore_index=True)
sample = sample.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

display(sample["analysis_group"].value_counts().rename("rows").to_frame())
print("Embedding rows:", len(sample))

## 2. Encode texts on GPU

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 64 if DEVICE == "cuda" else 16

encoder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
embeddings = encoder.encode(
    sample["text"].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

## 3. PCA and UMAP projections

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_xy = pca.fit_transform(embeddings)

reducer = umap.UMAP(
    n_neighbors=20,
    min_dist=0.15,
    metric="cosine",
    random_state=RANDOM_STATE,
)
umap_xy = reducer.fit_transform(embeddings)

projection = sample[["id", "label", "analysis_group", "source_document_id", "generator"]].copy()
projection[["pca_1", "pca_2"]] = pca_xy
projection[["umap_1", "umap_2"]] = umap_xy

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.scatterplot(data=projection, x="pca_1", y="pca_2", hue="analysis_group", alpha=0.65, s=35, ax=axes[0])
axes[0].set_title(f"PCA (explained variance: {pca.explained_variance_ratio_.sum():.1%})")

sns.scatterplot(data=projection, x="umap_1", y="umap_2", hue="analysis_group", alpha=0.65, s=35, ax=axes[1])
axes[1].set_title("UMAP of normalized sentence embeddings")

for ax in axes:
    ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "embedding_pca_umap.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. Group-centroid cosine similarity

In [ ]:
groups = sorted(sample["analysis_group"].unique())
centroids = []
for group in groups:
    centroid = embeddings[sample["analysis_group"].eq(group).to_numpy()].mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    centroids.append(centroid)

centroid_similarity = pd.DataFrame(
    cosine_similarity(np.vstack(centroids)),
    index=groups,
    columns=groups,
)
display(centroid_similarity.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(centroid_similarity, annot=True, fmt=".3f", cmap="mako", vmin=0, vmax=1)
plt.title("Cosine similarity between group centroids")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "centroid_cosine_similarity.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Nearest-neighbor inspection

In [ ]:
neighbors = NearestNeighbors(n_neighbors=6, metric="cosine").fit(embeddings)
distances, indices = neighbors.kneighbors(embeddings)

neighbor_rows = []
for row_index in range(len(sample)):
    for rank, (neighbor_index, distance) in enumerate(zip(indices[row_index, 1:], distances[row_index, 1:]), start=1):
        neighbor_rows.append(
            {
                "query_id": sample.iloc[row_index]["id"],
                "query_group": sample.iloc[row_index]["analysis_group"],
                "neighbor_id": sample.iloc[neighbor_index]["id"],
                "neighbor_group": sample.iloc[neighbor_index]["analysis_group"],
                "rank": rank,
                "cosine_similarity": 1 - distance,
            }
        )

neighbor_df = pd.DataFrame(neighbor_rows)
cross_group = neighbor_df.loc[neighbor_df["query_group"] != neighbor_df["neighbor_group"]]
display(cross_group.sort_values("cosine_similarity", ascending=False).head(30))

same_group_rate = (neighbor_df["query_group"] == neighbor_df["neighbor_group"]).mean()
print(f"Nearest-neighbor same-group rate across top 5: {same_group_rate:.2%}")
neighbor_df.to_csv(OUTPUT_DIR / "nearest_neighbors.csv", index=False)

## 6. Embedding classifier: internal vs external transfer

In [ ]:
hc3_mask = sample["analysis_group"].isin(["HC3 Human", "HC3 AI"]).to_numpy()
mp_mask = sample["analysis_group"].isin(["MedlinePlus Human", "MedlinePlus AI"]).to_numpy()

hc3_indices = np.flatnonzero(hc3_mask)
hc3_class_counts = sample.iloc[hc3_indices]["label"].value_counts()
print("HC3 rows selected for embedding classifier:")
display(hc3_class_counts.rename("rows").to_frame())
if len(hc3_class_counts) < 2:
    raise ValueError(
        "HC3 embedding subset must contain both human and AI labels. "
        f"Found: {hc3_class_counts.to_dict()}"
    )

train_idx, test_idx = train_test_split(
    hc3_indices,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=sample.iloc[hc3_indices]["label"],
)

classifier = LogisticRegression(max_iter=2_000, class_weight="balanced", random_state=RANDOM_STATE)
classifier.fit(embeddings[train_idx], sample.iloc[train_idx]["label"])

internal_pred = classifier.predict(embeddings[test_idx])
print("HC3 embedding holdout accuracy:", round(accuracy_score(sample.iloc[test_idx]["label"], internal_pred), 4))
print(classification_report(sample.iloc[test_idx]["label"], internal_pred, zero_division=0))

mp_idx = np.flatnonzero(mp_mask)
external_pred = classifier.predict(embeddings[mp_idx])
external_true = sample.iloc[mp_idx]["label"]

print("MedlinePlus transfer accuracy:", round(accuracy_score(external_true, external_pred), 4))
print(classification_report(external_true, external_pred, zero_division=0))
print("Confusion matrix labels [ai, human]:")
print(confusion_matrix(external_true, external_pred, labels=["ai", "human"]))

## 7. Source separability diagnostic

In [ ]:
source_labels = sample["analysis_group"].to_numpy()
train_idx, test_idx = train_test_split(
    np.arange(len(sample)),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=source_labels,
)

source_classifier = LogisticRegression(max_iter=2_000, class_weight="balanced", random_state=RANDOM_STATE)
source_classifier.fit(embeddings[train_idx], source_labels[train_idx])
source_pred = source_classifier.predict(embeddings[test_idx])

print("Source/group prediction accuracy:", round(accuracy_score(source_labels[test_idx], source_pred), 4))
print(classification_report(source_labels[test_idx], source_pred, zero_division=0))
print("\nHigh source accuracy indicates strong domain structure that an AI detector can exploit accidentally.")

## 8. Export embeddings and projections

In [ ]:
projection.to_csv(OUTPUT_DIR / "embedding_projection.csv", index=False)
np.savez_compressed(
    OUTPUT_DIR / "sentence_embeddings.npz",
    embeddings=embeddings,
    ids=sample["id"].to_numpy(),
    groups=sample["analysis_group"].to_numpy(),
    labels=sample["label"].to_numpy(),
)

import shutil
archive = shutil.make_archive("/content/medical_embedding_eda_outputs", "zip", OUTPUT_DIR)
print("Created:", archive)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

## Interpretation

Clear clusters do not prove that embeddings encode AI authorship. They may encode source,
genre, length, formatting, or generator family. Compare internal HC3 accuracy with external
MedlinePlus transfer and source-classification accuracy before claiming generalization.